# 09 · 기존 ACDC 증강 실험 재평가 — mirror TTA OFF
학습 없이 `spatial / quality / intensity / none`의 저장된 final·best checkpoint를 불러와 fold 0 validation 40개 영상을 다시 추론합니다.

- mirror TTA OFF, sliding window step 0.5, Gaussian weighting ON, 추가 후처리 없음.
- 모든 모델에 동일한 validation 환자 분할·원본 영상·지표 계산을 적용합니다.
- 기존 학습·TTA ON 결과를 덮어쓰지 않습니다. 새 `augmentation_reevaluation/OUTPUT_ID`에 저장합니다.
- 기존 `runs/<실험ID>`의 체크포인트, plans.json, dataset.json, config.snapshot.json, splits_final.json과 `nnUNet_raw/Dataset027_ACDC`가 필요합니다.
- 전처리 재실행·Swin·VoCo 가중치는 필요 없습니다. GPU 서버에서 실행하세요.


## 1. 초기화 및 실험 선택

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
os.environ['PYTORCH_CUDA_ALLOC_CONF']='expandable_segments:True'
REPO=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'nnunetv2').is_dir() and (p/'pyproject.toml').exists()),None)
if REPO is None: raise RuntimeError('저장소 안에서 노트북을 여세요.')
if sys.platform!='linux': raise RuntimeError('실제 추론은 Linux GPU 서버에서만 실행하세요.')
EXPERIMENT=REPO/'experiments/acdc'
STORAGE_ROOT=REPO.parent/'acdc-data'
GPU='0'
OUTPUT_ID='acdc-aug250-no-tta-v1'
AUG_SUITE_ID='acdc-aug250-v1'
RUNS={theme:f'{AUG_SUITE_ID}-{theme}' for theme in ['spatial','quality','intensity','none']}
# 선택 사항: 1000 epoch baseline도 함께 재평가 (학습량 차이는 결과에 표시)
# RUNS['baseline_1000']='acdc-baseline-v1'
CHECKPOINTS=['final','best']  # 빠르게 best만 보려면 ['best']
INSTALL_PACKAGES=False
RUN_EVALUATION=False
OUTPUT=STORAGE_ROOT/'augmentation_reevaluation'/OUTPUT_ID
print('결과 저장:',OUTPUT)
print('선택한 실험:',RUNS)


## 2. 의존성 — 기존 nnU-Net 커널이면 생략
빈 Python 3.11 venv에서는 INSTALL_PACKAGES=True로 실행하세요. 기존 torch가 있다면 보존합니다. 설치 후 커널 재시작 → INSTALL_PACKAGES=False → 1절부터 실행하세요.

In [ ]:
if INSTALL_PACKAGES:
    import importlib.metadata
    if sys.prefix==sys.base_prefix: raise RuntimeError('venv 커널을 선택하세요.')
    pip=[sys.executable,'-m','pip']
    if importlib.util.find_spec('torch') is None:
        subprocess.run(pip+['install','--no-cache-dir','torch==2.8.0',
            '--index-url','https://download.pytorch.org/whl/cu126'],check=True)
    pin='torch=='+importlib.metadata.version('torch')
    subprocess.run(pip+['install','--no-cache-dir',pin,'-e',str(REPO),
        '-r',str(EXPERIMENT/'requirements-server.txt')],check=True)
    subprocess.run(pip+['check'],check=True)
    print('커널을 재시작하고 1절부터 실행하세요.')
else:
    print('기존 환경 사용')


## 3. 체크포인트·환자 분할 확인 — 아직 추론하지 않음

In [ ]:
script=EXPERIMENT/'analysis/reevaluate_augmentation_no_tta.py'
if not script.exists(): raise FileNotFoundError('노트북과 새 evaluation 스크립트를 함께 복사하세요.')
sys.path.insert(0,str(EXPERIMENT/'analysis'))
spec=importlib.util.spec_from_file_location('acdc_aug_no_tta',script)
evaluator=importlib.util.module_from_spec(spec)
spec.loader.exec_module(evaluator)
config=dict(storage_root=str(STORAGE_ROOT.resolve()),gpu=GPU,output_id=OUTPUT_ID,
            runs=RUNS,checkpoints=CHECKPOINTS,execution_enabled=True)
records,cases=evaluator.models_and_cases(config)
for r in records: print(r['name'], '→', r['checkpoint'])
print('공통 validation 영상:',len(cases))


## 4. 재추론 실행
`RUN_EVALUATION=True`로 변경하고 실행하세요. 학습 없이 선택한 checkpoint만 순차 평가합니다.
다른 GPU 작업은 먼저 종료하세요. 이 셀은 완료될 때까지 실행되므로 서버·커널을 유지하세요.
중단되면 같은 설정으로 다시 실행합니다. manifest와 유효한 예측을 확인해 완료된 영상을 재사용합니다.
입력/코드/환경이 바뀌거나 전부 다시 추론하려면 새 OUTPUT_ID를 사용하세요.


In [ ]:
RUN_EVALUATION=False
if RUN_EVALUATION:
    config_file=STORAGE_ROOT/'evaluation_configs'/f'{OUTPUT_ID}.json'
    evaluator.write_json(config_file,config)
    env=dict(os.environ,PYTHONUNBUFFERED='1',PYTHONDONTWRITEBYTECODE='1')
    subprocess.run([sys.executable,'-u','-B',str(script),'--config',str(config_file)],
                   cwd=REPO,env=env,check=True)
else:
    print('RUN_EVALUATION=True로 바꾸면 재추론합니다.')


## 5. 결과 표·그래프
Mean Dice/IoU: 영상별 각 클래스 점수 평균 → RV/MLV/LVC 평균. 둘 다 없는 클래스는 NaN으로 두고 평균에서 제외하며 실제 0점은 유지합니다. final과 best를 구분해 비교합니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
result=OUTPUT/'comparison.json'
if result.exists():
    report=json.loads(result.read_text())
    print('전체 평가 완료:',report['complete'])
    table=pd.DataFrame(report['results'])
    display(table)
    fig,axes=plt.subplots(1,2,figsize=(13,4))
    for ax,metric,title in zip(axes,['dice_percent','miou_percent'],['Mean Dice (%)','Mean IoU (%)']):
        table.pivot(index='experiment',columns='checkpoint',values=metric).plot.bar(ax=ax)
        ax.set(title=title+' — mirror TTA OFF',ylabel=title,xlabel='')
        ax.tick_params(axis='x',rotation=15)
    fig.tight_layout()
    fig.savefig(OUTPUT/'comparison.png',dpi=180)
    plt.show()
    print('결과:',OUTPUT)
else:
    print('아직 평가 결과가 없습니다.')


## 6. 영상별 결과 확인 — 선택 사항

In [ ]:
THEME='none'
CHECKPOINT='best'
file=OUTPUT/THEME/CHECKPOINT/'case-metrics.csv'
if file.exists():
    cases_table=pd.read_csv(file)
    display(cases_table)
    display(cases_table.groupby('case')[['dice','iou']].mean().sort_values('dice').head(10))
else:
    print('해당 평가 완료 후 확인하세요:',file)
